# Calculate PSI for colonies using buffer (3 km) [25 August 2020]

## Import modules, set constants

In [ ]:
import os
import pickle
from importlib import reload
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon, box
import spatial_index_utils
from spatial_index_utils import reproject_gdf, calc_all_services_buffer

reload(spatial_index_utils)

# WGS 84 / Delhi
epsg_code = 7760

# size of max buffer: 1km = 1000 meters
buffer_size = 3000

## Import colonies shapefile

In [ ]:
with open('colonies_touch_nbrs25Aug2020.pkl', 'rb') as f:
    colonies = pickle.load(f)
    
colonies.head()

In [ ]:
colonies = colonies.drop(columns=['canal', 'railway', 'drain', 'barrier', 'nbrs_touch', 'nbrs_dist_touch'])

In [ ]:
colonies.head()

In [ ]:
colonies.crs

## Create buffer and set this to new geometry

In [ ]:
colonies['buffer'] = colonies.buffer(distance=buffer_size)

In [ ]:
colonies['geometry'][1000]

In [ ]:
colonies['buffer'][1000]

In [ ]:
colonies.head()

In [ ]:
colonies_buffer = colonies.copy()

In [ ]:
colonies_buffer = colonies_buffer.drop(columns=['geometry']).rename(columns={'buffer':'geometry'})

In [ ]:
colonies_buffer.head()

In [ ]:
colonies.head()

In [ ]:
colonies_buffer.crs

In [ ]:
colonies_buffer.head()

## Import services shapefiles

In [ ]:
# Define filepaths

services_dir = os.path.join('shapefiles', 'Spatial_Index_GIS', 'Public Services')

bank_fp = os.path.join(services_dir, 'Banking', 'Banking.shp')
health_fp = os.path.join(services_dir, 'Health', 'Health.shp')
road_fp = os.path.join(services_dir, 'Major Road', 'Road.shp')
police_fp = os.path.join(services_dir, 'Police', 'Police Station.shp')
ration_fp = os.path.join(services_dir, 'Ration', 'Ration.shp')
school_fp = os.path.join(services_dir, 'School', 'schools7760.shp')
transport_fp = os.path.join(services_dir, 'Transport', 'Transport.shp')

# boundary of Delhi
delhi_bounds_filepath = os.path.join('shapefiles', 'delhi_bounds_buffer.shp')

# Check that all filepaths exist
filepath_list = [bank_fp, health_fp, road_fp, police_fp, ration_fp, school_fp, transport_fp, delhi_bounds_filepath]

for filepath in filepath_list:
    if not os.path.exists(filepath):
        print('{} does not exist'.format(filepath))
        
# Import services
bank = gpd.read_file(bank_fp)
health = gpd.read_file(health_fp)
road = gpd.read_file(road_fp)
police = gpd.read_file(police_fp)
ration = gpd.read_file(ration_fp)
school = gpd.read_file(school_fp)
transport = gpd.read_file(transport_fp)

In [ ]:
bank.crs == health.crs == road.crs == police.crs == ration.crs == school.crs == transport.crs == colonies_buffer.crs

## Define Point and Line Services

In [ ]:
# Define all point services as dictionary
# makes it easier to calculate all point
# services with one function
point_services = {'bank': bank,
                  'health': health,
                  'police': police,
                  'ration': ration,
                  'school': school,
                  'transport': transport}

line_services = {'road': road}

## Colonies with Buffer Service Index (population size)

In [ ]:
colonies_buffer_idx_popsize = calc_all_services_buffer(polygon_gdf = colonies_buffer, 
                                               point_services = point_services, 
                                               line_services = line_services, 
                                               calc_pop_density = False,
                                               epsg_code = epsg_code)

In [ ]:
colonies_buffer_idx_popsize.head()

## Colonies with Buffer Service Index (population density)

In [ ]:
colonies_buffer_idx_popdensity = calc_all_services_buffer(polygon_gdf = colonies_buffer, 
                                               point_services = point_services, 
                                               line_services = line_services, 
                                               calc_pop_density = True,
                                               epsg_code = epsg_code)

colonies_buffer_idx_popdensity.head()

## Visualizing Results

In [ ]:
colonies_buffer_idx['ration_idx'].plot(kind='hist')

In [ ]:
colonies_buffer_idx['bank_idx'].plot(kind='hist')

In [ ]:
colonies_buffer_idx['health_idx'].plot(kind='hist')

In [ ]:
colonies_buffer_idx['police_idx'].plot(kind='hist')

In [ ]:
colonies_buffer_idx['transport_idx'].plot(kind='hist')

In [ ]:
colonies_buffer_idx['road_idx'].plot(kind='hist')

In [ ]:
colonies_buffer_idx['school_idx'].plot(kind='hist')

## Save Files

In [ ]:
colonies_buffer_idx_popsize.drop(columns=['centroid']).to_file('colonies_psi_3km_buffer_popsize.shp')
colonies_buffer_idx_popsize.to_csv('colonies_psi_3km_buffer_popsize.csv')

In [ ]:
colonies_buffer_idx_popdensity.drop(columns=['centroid']).to_file('colonies_psi_3km_buffer_popdensity.shp')
colonies_buffer_idx_popdensity.to_csv('colonies_psi_3km_buffer_popdensity.csv')

## Open File and Visualize Results

In [ ]:
colonies_buffer_idx_popsize = gpd.read_file('colonies_psi_3km_buffer_popsize.shp')

In [ ]:
colonies_buffer_idx_popsize['road_idx'].plot(kind='hist')

In [ ]:
colonies_buffer_idx_popdensity = gpd.read_file('colonies_psi_3km_buffer_popdensity.shp')

In [ ]:
colonies_buffer_idx_popdensity['road_idx'].plot(kind='hist')